In [1]:
# Agent = Planner + Tool Executor + Reasoner

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import json

In [18]:
llm = ChatOpenAI(api_key="not-needed",
                base_url="http://localhost:1234/v1",
                model="mistral")


@tool(description="Track daily walking goal of 10,000 steps")
def walking_10000_steps() -> dict:
    return {
        "action": "Walking 10,000 steps daily",
        "priority": 1,
        "expected_time": "30 minutes",
        "dependency": []
    }

@tool(description="Track strength training routine 3 times per week.")
def strength_training() -> dict:
    return {
        "action": "Strength training 3x/week",
        "priority": 2,
        "expected_time": "30 minutes",
        "dependency": []
    }

@tool
def track_calories() -> dict:
    """Track daily calorie intake."""
    return {
        "action": "Track calories daily",
        "priority": 3,
        "expected_time": "10 minutes per day",
        "dependency": []
    }

@tool
def sleep_routine() -> dict:
    """Ensure 7-8 hours of sleep each night."""
    return {
        "action": "Sleep 7-8 hours nightly",
        "priority": 4,
        "expected_time": "7-8 hours per night",
        "dependency": []
    }

@tool
def weekly_review() -> dict:
    """Review weekly progress on all subgoals."""
    return {
        "action": "Weekly progress review",
        "priority": 5,
        "expected_time": "30 minutes weekly",
        "dependency": []
    }

@tool
def tell_a_joke() -> dict:
    """This is used for creating a joke"""
    return {
        "action": "Weekly progress review",
        "priority": 5,
        "expected_time": "30 minutes weekly",
        "dependency": []
    }

In [22]:
tool_descriptions = [
      {
        "name": "tell_a_joke",
        "description": "This is used for creating a joke"
    },
    {
        "name": "walking_10000_steps",
        "description": "Helps with daily step goals for fat loss"
    },
    {
        "name": "strength_training",
        "description": "Provides strength training routine"
    },
    {
        "name": "track_calories",
        "description": "Tracks daily calorie intake"
    },
    {
        "name": "sleep_routine",
        "description": "Ensures proper sleep schedule"
    },
    {
        "name": "weekly_review",
        "description": "Reviews weekly progress"
    }
   
]

def planner_llm(user_query, tool_description, llm):
    tools_text = "\n".join(
    [f"- {tool['name']}: {tool['description']}" for tool in tool_description]
)
    prompt = f'''
You are a Planner

User request:
{user_query}

Available tools:
{tools_text}

Return ONLY a JSON list of tool names that are relevant.
Example:
["walking_10000_steps", "strength_training"]

Do not explain anything
'''
    response = llm.invoke(prompt)
    content = response.content.strip()

    try:
        selected_tools = json.loads(content)
    
    except Exception:
        print("Parsing failed, fallback to all tools")
        selected_tools = [tool["name"] for tool in tool_description]

    return selected_tools



In [23]:
tool_map = {
    "tell_a_joke": tell_a_joke,
    "walking_10000_steps": walking_10000_steps,
    "strength_training": strength_training,
    "track_calories": track_calories,
    "sleep_routine": sleep_routine,
    "weekly_review": weekly_review,
}

selected = planner_llm(user_query="Create a fat loss and strength plan",
                       tool_description=tool_descriptions,
                       llm=llm)

In [24]:
results = []
selected

['walking_10000_steps',
 'strength_training',
 'track_calories',
 'sleep_routine',
 'weekly_review']